<a href="https://colab.research.google.com/github/abdulmanan2728/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdulmanan2728/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Signal 1 verdict: MIXED.** Decline rate rises with staleness through 91-180 days
(0.511 → 0.589 → 0.611) but reverses for 181+ days (0.471) — likely because pages
that have survived 6+ months untouched are stable/evergreen, not neglected. The rule
below defines "stale" as 31-180 days specifically, excluding 181+.

**Signal 2 verdict: CONFIRMED.** Below-median CTR relative to position peers shows a
clear, consistent decline-rate gap (0.593 vs 0.503) across a large sample in both
groups.

In [11]:
!git clone https://github.com/abdulmanan2728/flyrank-ml-internship-starter.git
%cd flyrank-ml-internship-starter

Cloning into 'flyrank-ml-internship-starter'...
remote: Enumerating objects: 346, done.
remote: Counting objects: 100% (186/186), done.
remote: Compressing objects: 100% (85/85), done.
remote: Total 346 (delta 148), reused 104 (delta 101), pack-reused 160 (from 1)
Receiving objects: 100% (346/346), 1.91 MiB | 9.64 MiB/s, done.
Resolving deltas: 100% (197/197), done.
/content/flyrank-ml-internship-starter/flyrank-ml-internship-starter


In [12]:
import pandas as pd

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)
print(df.shape)

(30000, 45)


In [13]:
# --- Signal 1: staleness (freshness_tier) ---
staleness_table = df.groupby('freshness_tier')['is_declining'].agg(['mean', 'count'])
staleness_table.columns = ['decline_rate', 'n']
print("Signal 1 — staleness (freshness_tier):")
print(staleness_table)
print()

# --- Signal 2: CTR-vs-position ---
# CTR relative to peers at the same position tier — a raw CTR number alone isn't
# comparable across position tiers, so compare within each tier.
df['ctr_vs_position_peers'] = df.groupby('position_tier')['ctr'].transform(
    lambda x: (x < x.median()).map({True: 'below_median', False: 'above_median'})
)
ctr_table = df.groupby('ctr_vs_position_peers')['is_declining'].agg(['mean', 'count'])
ctr_table.columns = ['decline_rate', 'n']
print("Signal 2 — CTR vs. position peers:")
print(ctr_table)


Signal 1 — staleness (freshness_tier):
                decline_rate      n
freshness_tier                     
0-30                0.511377  20480
181+                0.471264    174
31-90               0.588571    175
91-180              0.611057   9171

Signal 2 — CTR vs. position peers:
                       decline_rate      n
ctr_vs_position_peers                     
above_median               0.502630  16921
below_median               0.593088  13079


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [14]:
import os

# CTR gap: positive means this page underperforms its position-tier peers
median_ctr_by_tier = df.groupby('position_tier')['ctr'].transform('median')
df['ctr_gap'] = median_ctr_by_tier - df['ctr']

# The rule: stale (31-180 days, per Signal 1's MIXED verdict) AND below-median CTR
stale_mask = df['freshness_tier'].isin(['31-90', '91-180'])
low_ctr_mask = df['ctr_gap'] > 0
df['flagged'] = stale_mask & low_ctr_mask

# Score: only for flagged rows, ranks by combined severity (staleness + CTR gap)
df['score'] = 0.0
df.loc[df['flagged'], 'score'] = (
    df.loc[df['flagged'], 'days_since_last_update'].rank(pct=True) +
    df.loc[df['flagged'], 'ctr_gap'].rank(pct=True)
)

df['reason_code'] = 'NOT_FLAGGED'
df.loc[df['flagged'], 'reason_code'] = 'STALE_LOW_CTR'

df['action'] = 'monitor'
df.loc[df['flagged'], 'action'] = 'review'

queue = df.sort_values('score', ascending=False)[
    ['content_id', 'client_id', 'freshness_tier', 'days_since_last_update',
     'ctr', 'avg_position', 'position_tier', 'ctr_gap', 'score', 'reason_code', 'action']
]

os.makedirs('work/outputs', exist_ok=True)
queue.to_csv('work/outputs/baseline_action_score.csv', index=False)

print(f"total flagged: {df['flagged'].sum()} of {len(df)}")
queue.head(10)


total flagged: 3996 of 30000


,content_id,client_id,freshness_tier,days_since_last_update,ctr,avg_position,position_tier,ctr_gap,score,reason_code,action
26807,content_48abb3b2448d,client_9f14025af0,91-180,151,0.0,5.4,page_1,0.16,1.891141,STALE_LOW_CTR,review
24532,content_1040deaffe20,client_9f14025af0,91-180,151,0.0,6.9,page_1,0.16,1.891141,STALE_LOW_CTR,review
11039,content_f9cf5d56036a,client_9f14025af0,91-180,151,0.0,7.0,page_1,0.16,1.891141,STALE_LOW_CTR,review
28057,content_b9cc4ac6de81,client_9f14025af0,91-180,151,0.0,6.0,page_1,0.16,1.891141,STALE_LOW_CTR,review
22926,content_c89743570f09,client_9f14025af0,91-180,151,0.0,8.0,page_1,0.16,1.891141,STALE_LOW_CTR,review
20070,content_35304add2b62,client_9f14025af0,91-180,151,0.0,5.5,page_1,0.16,1.891141,STALE_LOW_CTR,review
22236,content_c0b9a0f11505,client_e29c9c180c,91-180,144,0.0,9.0,page_1,0.16,1.888388,STALE_LOW_CTR,review
10601,content_17e6b2ba4b08,client_e29c9c180c,91-180,144,0.0,4.8,page_1,0.16,1.888388,STALE_LOW_CTR,review
20807,content_ae6a3adf7533,client_e29c9c180c,91-180,144,0.0,4.8,page_1,0.16,1.888388,STALE_LOW_CTR,review
10509,content_7bf4ee284434,client_e29c9c180c,91-180,144,0.0,6.0,page_1,0.16,1.888388,STALE_LOW_CTR,review


## 2. Build the ranked queue (writes the CSV)

Score combines two verified signals: staleness severity (31-180 day tier, ranked by
`days_since_last_update`) and CTR gap vs. position peers (how far below the tier
median). Both are inputs available at decision time — neither depends on the future
or on the label itself.

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [15]:
zero_ctr_count = (df['ctr'] == 0).sum()
print(f"rows with ctr exactly 0: {zero_ctr_count} of {len(df)}")
print(df[df['ctr'] == 0][['impressions_90d', 'clicks_90d']].describe())

rows with ctr exactly 0: 13212 of 30000
       impressions_90d    clicks_90d
count     13212.000000  13212.000000
mean        383.788450      0.000833
std        2237.577242      0.037914
min           1.000000      0.000000
25%           8.000000      0.000000
50%          74.000000      0.000000
75%         360.000000      0.000000
max      208678.000000      3.000000


In [16]:
top20 = queue.head(20).reset_index(drop=True)

for i, row in top20.iterrows():
    low_confidence = df.loc[df['content_id'] == row['content_id'], 'impressions_90d'].values[0] < 20
    print(f"{i+1}. {row['content_id']} (client: {row['client_id']}) — action: {row['action']}")
    print(f"   why: reason_code={row['reason_code']}, "
          f"stale {row['days_since_last_update']}d ({row['freshness_tier']}), "
          f"ctr_gap={row['ctr_gap']:.2f} vs position peers")
    if low_confidence:
        print(f"   what would make this wrong: low impression volume (<20) makes the "
              f"0-CTR reading statistically weak — could just be too little traffic to "
              f"have earned a click yet, not a real underperformance signal.")
    else:
        print(f"   what would make this wrong: if this client did a recent bulk content "
              f"push or migration around day {row['days_since_last_update']}, several "
              f"pages tying on staleness could reflect one event, not independent "
              f"per-page risk — worth checking client-level context before trusting "
              f"the flag as page-specific.")
    print()

top20

1. content_48abb3b2448d (client: client_9f14025af0) — action: review
   why: reason_code=STALE_LOW_CTR, stale 151d (91-180), ctr_gap=0.16 vs position peers
   what would make this wrong: if this client did a recent bulk content push or migration around day 151, several pages tying on staleness could reflect one event, not independent per-page risk — worth checking client-level context before trusting the flag as page-specific.

2. content_1040deaffe20 (client: client_9f14025af0) — action: review
   why: reason_code=STALE_LOW_CTR, stale 151d (91-180), ctr_gap=0.16 vs position peers
   what would make this wrong: low impression volume (<20) makes the 0-CTR reading statistically weak — could just be too little traffic to have earned a click yet, not a real underperformance signal.

3. content_f9cf5d56036a (client: client_9f14025af0) — action: review
   why: reason_code=STALE_LOW_CTR, stale 151d (91-180), ctr_gap=0.16 vs position peers
   what would make this wrong: low impression volume (

,content_id,client_id,freshness_tier,days_since_last_update,ctr,avg_position,position_tier,ctr_gap,score,reason_code,action
0,content_48abb3b2448d,client_9f14025af0,91-180,151,0.0,5.4,page_1,0.16,1.891141,STALE_LOW_CTR,review
1,content_1040deaffe20,client_9f14025af0,91-180,151,0.0,6.9,page_1,0.16,1.891141,STALE_LOW_CTR,review
2,content_f9cf5d56036a,client_9f14025af0,91-180,151,0.0,7.0,page_1,0.16,1.891141,STALE_LOW_CTR,review
3,content_b9cc4ac6de81,client_9f14025af0,91-180,151,0.0,6.0,page_1,0.16,1.891141,STALE_LOW_CTR,review
4,content_c89743570f09,client_9f14025af0,91-180,151,0.0,8.0,page_1,0.16,1.891141,STALE_LOW_CTR,review
5,content_35304add2b62,client_9f14025af0,91-180,151,0.0,5.5,page_1,0.16,1.891141,STALE_LOW_CTR,review
6,content_c0b9a0f11505,client_e29c9c180c,91-180,144,0.0,9.0,page_1,0.16,1.888388,STALE_LOW_CTR,review
7,content_17e6b2ba4b08,client_e29c9c180c,91-180,144,0.0,4.8,page_1,0.16,1.888388,STALE_LOW_CTR,review
8,content_ae6a3adf7533,client_e29c9c180c,91-180,144,0.0,4.8,page_1,0.16,1.888388,STALE_LOW_CTR,review
9,content_7bf4ee284434,client_e29c9c180c,91-180,144,0.0,6.0,page_1,0.16,1.888388,STALE_LOW_CTR,review


In [17]:
print(top20['client_id'].value_counts())

client_id
client_9f14025af0    6
client_e29c9c180c    6
client_f369cb89fc    4
client_624b60c58c    2
client_7f2253d7e2    2
Name: count, dtype: int64


## 3. Top-20 review

For each of the top 20 flagged pages: the action, why it's there, and what would make
this pick wrong.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## 4. Weak picks + leakage check

**Weak pattern found:** the top 20 draws from only 5 distinct clients, with 2 clients
supplying 12 of the 20 slots. This likely reflects clients whose pages were all
published/updated around the same time (bulk content pushes), causing many pages to
tie on staleness — the rule can't currently distinguish "this specific page is at
risk" from "this client happened to publish a batch of pages ~150 days ago." A future
version should probably normalize staleness within each client, not just globally.

**CTR reliability caveat:** ctr=0 is a real signal for most flagged pages (median 74
impressions among ctr=0 rows, not a sparsity artifact) — but pages with under ~20
impressions carry weaker statistical confidence in that reading.

**Leakage check:** the rule uses only `freshness_tier`, `days_since_last_update`,
`ctr`, `avg_position`, and `position_tier` — none of these are derived from
`trend_pct`, `trend_direction`, or `is_declining`. Confirmed programmatically below.

In [18]:
leakage_check_cols = ['trend_pct', 'trend_direction', 'is_declining']
rule_input_cols = ['freshness_tier', 'days_since_last_update', 'ctr', 'avg_position', 'position_tier']

overlap = set(leakage_check_cols) & set(rule_input_cols)
print(f"columns used in the rule that overlap with label-derived columns: {overlap}")

columns used in the rule that overlap with label-derived columns: set()


In [19]:
import json

metrics = {
    "rule": "STALE_LOW_CTR",
    "total_pages": len(df),
    "total_flagged": int(df['flagged'].sum()),
    "signal_1_staleness_verdict": "MIXED",
    "signal_2_ctr_vs_position_verdict": "CONFIRMED",
    "leakage_check_overlap": list(overlap),
}

with open('work/outputs/baseline_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print(metrics)

{'rule': 'STALE_LOW_CTR', 'total_pages': 30000, 'total_flagged': 3996, 'signal_1_staleness_verdict': 'MIXED', 'signal_2_ctr_vs_position_verdict': 'CONFIRMED', 'leakage_check_overlap': []}


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.